In [40]:
from pathlib import Path
import os

# resolve project root (assumes notebook is in notebooks/)
PROJECT_ROOT = Path.cwd().parent
print("Project root:", PROJECT_ROOT)

# optional: make it the working directory
os.chdir(PROJECT_ROOT)


Project root: /Users/gretaharsanyi/Suli/Cognitive Science Masters/1st Semester/Natural Language Processing/Exam/nlp-exam-tactile-classification-cogscimsc


In [38]:
# imports and loading data

import json
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report




SAMPLES_PATH = Path("..") / "data/generated/raw/samples.jsonl"  # notebook is in notebooks/
# If you already chdir(".."), then use: Path("data/generated/raw/samples.jsonl")

rows = []
with open(SAMPLES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df.shape, df.columns


((6892, 11),
 Index(['job_id', 'provider', 'model', 'prompt_id', 'text', 'labels', 'literal',
        'specificity', 'consistent', 'anchor_regime', 'anchors_planned'],
       dtype='str'))

In [11]:
# parsing multi-label targets
# converting columns into list format 

df["label_list"] = df["labels"].str.split("+")
df["label_list"].head()


0    [temperature, vibration, pressure]
1    [temperature, vibration, pressure]
2    [temperature, vibration, pressure]
3               [temperature, pressure]
4               [temperature, pressure]
Name: label_list, dtype: object

In [12]:
# sanity checking unique labels

sorted({lab for labs in df["label_list"] for lab in labs})


['nociception', 'pressure', 'temperature', 'vibration']

In [13]:
# train/val/test splitting by job_id - note on why should be added in paper!!

job_ids = df["job_id"].unique()

rng = np.random.default_rng(42)
rng.shuffle(job_ids)

n = len(job_ids)
train_ids = set(job_ids[: int(0.8 * n)])
val_ids   = set(job_ids[int(0.8 * n): int(0.9 * n)])
test_ids  = set(job_ids[int(0.9 * n):])

def assign_split(j):
    if j in train_ids: return "train"
    if j in val_ids:   return "val"
    return "test"

df["split"] = df["job_id"].map(assign_split)

df["split"].value_counts(), df["split"].value_counts(normalize=True)


/var/folders/9z/8mkwm57j4rq2mzhmvzjhn2n80000gn/T/ipykernel_13399/2601301729.py:6: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(job_ids)


(split
 train    5533
 val       690
 test      669
 Name: count, dtype: int64,
 split
 train    0.802815
 val      0.100116
 test     0.097069
 Name: proportion, dtype: float64)

In [14]:
# sanity checkk - works!

df.groupby("split")["label_list"].apply(lambda s: pd.Series([x for xs in s for x in xs]).value_counts())

split             
test   vibration       354
       pressure        270
       nociception     249
       temperature     246
train  pressure       2272
       temperature    2265
       nociception    2262
       vibration      2141
val    vibration       300
       nociception     291
       temperature     279
       pressure        252
Name: label_list, dtype: int64

In [15]:
# vectorising text and binarising labels 

X_text = df["text"].values
y_lists = df["label_list"].values

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(y_lists)

mlb.classes_, Y.shape


(array(['nociception', 'pressure', 'temperature', 'vibration'],
       dtype=object),
 (6892, 4))

In [16]:
# training baseline TF-IDF + OvR logistic regression

# split indices
train_mask = df["split"].eq("train").values
val_mask   = df["split"].eq("val").values
test_mask  = df["split"].eq("test").values

tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
)

X_train = tfidf.fit_transform(X_text[train_mask])
X_val   = tfidf.transform(X_text[val_mask])
X_test  = tfidf.transform(X_text[test_mask])

clf = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        solver="liblinear",
    )
)

clf.fit(X_train, Y[train_mask])


,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...r='liblinear')
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=

In [19]:
# eval (overall and per-label eval)


Y_val_pred  = clf.predict(X_val)
Y_test_pred = clf.predict(X_test)

print("VAL micro F1:", f1_score(Y[val_mask], Y_val_pred, average="micro"))
print("VAL macro F1:", f1_score(Y[val_mask], Y_val_pred, average="macro"))

print("\nTEST micro F1:", f1_score(Y[test_mask], Y_test_pred, average="micro"))
print("TEST macro F1:", f1_score(Y[test_mask], Y_test_pred, average="macro"))

print("\nPer-label report (TEST):")
print(classification_report(Y[test_mask], Y_test_pred, target_names=mlb.classes_))


VAL micro F1: 0.9025304592314901
VAL macro F1: 0.9015823620838187

TEST micro F1: 0.9059266227657573
TEST macro F1: 0.901205501851932

Per-label report (TEST):
              precision    recall  f1-score   support

 nociception       0.96      0.86      0.91       249
    pressure       0.96      0.86      0.91       270
 temperature       0.91      0.79      0.84       246
   vibration       0.98      0.91      0.94       354

   micro avg       0.96      0.86      0.91      1119
   macro avg       0.95      0.86      0.90      1119
weighted avg       0.96      0.86      0.91      1119
 samples avg       0.95      0.89      0.91      1119



/Users/gretaharsanyi/Suli/Cognitive Science Masters/1st Semester/Natural Language Processing/Exam/nlp-exam-tactile-classification-cogscimsc/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [21]:
leak_terms = ["nociception", "temperature", "pressure", "vibration", "dataset", "label"]
for t in leak_terms:
    n = df["text"].str.lower().str.contains(t).sum()
    print(t, n)


nociception 0
temperature 26
pressure 758
vibration 539
dataset 0
label 0


In [22]:
# testing whether the appearance of the labels drives  F1 scores

import re
import numpy as np
from sklearn.metrics import f1_score

def mask_label_words(text):
    t = text
    # mask common label-ish tokens
    t = re.sub(r"\bpressure\b", "___", t, flags=re.IGNORECASE)
    t = re.sub(r"\bvibration(s)?\b", "___", t, flags=re.IGNORECASE)
    t = re.sub(r"\btemperature\b", "___", t, flags=re.IGNORECASE)
    return t

# Build masked X_test
masked_test_text = np.array([mask_label_words(t) for t in df.loc[test_mask, "text"].values])
X_test_masked = tfidf.transform(masked_test_text)

Y_test_pred_masked = clf.predict(X_test_masked)

print("Original TEST micro F1:", f1_score(Y[test_mask], Y_test_pred, average="micro"))
print("Masked   TEST micro F1:", f1_score(Y[test_mask], Y_test_pred_masked, average="micro"))


Original TEST micro F1: 0.9059266227657573
Masked   TEST micro F1: 0.8831664282308059


Explicit modality words help a bit, but they are NOT the main driver of performance.

for paper: To assess whether classification performance was driven primarily by explicit modality tokens, we conducted an auxiliary evaluation in which common modality words (e.g., “pressure”, “vibration”) were masked at test time. Performance decreased modestly (micro F1 from 0.91 to 0.88), indicating that while explicit lexical cues contribute to classification, the majority of predictive signal arises from broader linguistic patterns associated with tactile experience.

In [24]:
import numpy as np

job_ids = df["job_id"].unique()
rng = np.random.default_rng(42)
rng.shuffle(job_ids)

n = len(job_ids)
train_ids = set(job_ids[: int(0.8*n)])
val_ids   = set(job_ids[int(0.8*n): int(0.9*n)])

df["split"] = df["job_id"].apply(
    lambda j: "train" if j in train_ids else ("val" if j in val_ids else "test")
)

df["split"].value_counts()


/var/folders/9z/8mkwm57j4rq2mzhmvzjhn2n80000gn/T/ipykernel_13399/2419271823.py:5: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(job_ids)


split
train    5533
val       690
test      669
Name: count, dtype: int64

In [26]:
# performance breakdown by experimental axes 


test_df = df[df["split"] == "test"].copy()

# predictions for test rows aligned with test_df order
test_df["pred_list"] = list(mlb.inverse_transform(Y_test_pred))
test_df["true_list"] = list(mlb.inverse_transform(Y[test_mask]))

def f1_for_subset(mask):
    idx = test_df.index[mask]
    if len(idx) == 0:
        return np.nan
    # re-vectorize subset into matrices aligned to predictions
    sub_true = mlb.transform(test_df.loc[idx, "true_list"])
    sub_pred = mlb.transform(test_df.loc[idx, "pred_list"])
    return f1_score(sub_true, sub_pred, average="micro")

# Example: by anchor_regime
for regime in ["strict", "paraphrase", "drift"]:
    m = test_df["anchor_regime"].eq(regime)
    print(regime, "micro F1:", f1_for_subset(m))

# Example: literal vs metaphorical
for lit in [0, 1]:
    m = test_df["literal"].eq(lit)
    print("literal" if lit==1 else "metaphor", "micro F1:", f1_for_subset(m))


strict micro F1: 0.8772845953002611
paraphrase micro F1: 0.9251497005988024
drift micro F1: 0.9190751445086706
metaphor micro F1: 0.8835930339138405
literal micro F1: 0.9294685990338164


### FULL EVAL BELOW

In [27]:
# one unified eval table across all dimensions

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

test_mask = df["split"].eq("test").values
test_df = df.loc[test_mask].copy().reset_index(drop=True)

Y_test_true = Y[test_mask]
Y_test_pred = clf.predict(X_test)

test_df["true_list"] = list(mlb.inverse_transform(Y_test_true))
test_df["pred_list"] = list(mlb.inverse_transform(Y_test_pred))

test_df["k_mods"] = test_df["labels"].astype(str).str.split("+").apply(len)

def micro_f1_subset(mask):
    mask = np.array(mask)
    if mask.sum() == 0:
        return np.nan
    sub_true = mlb.transform(test_df.loc[mask, "true_list"])
    sub_pred = mlb.transform(test_df.loc[mask, "pred_list"])
    return f1_score(sub_true, sub_pred, average="micro", zero_division=0)

def summarise_by(col, values=None):
    out = []
    if values is None:
        values = sorted(test_df[col].dropna().unique().tolist())
    for v in values:
        m = test_df[col].eq(v)
        out.append({"axis": col, "value": v, "n": int(m.sum()), "micro_f1": micro_f1_subset(m)})
    return pd.DataFrame(out)

all_tables = []


In [29]:
# single-axis breakdowns
all_tables.append(summarise_by("anchor_regime", ["strict","paraphrase","drift"]))
all_tables.append(summarise_by("literal", [0,1]))
all_tables.append(summarise_by("specificity", [0,1]))
all_tables.append(summarise_by("consistent", [0,1]))
all_tables.append(summarise_by("k_mods", [1,2,3]))

axis_table = pd.concat(all_tables, ignore_index=True)
axis_table



,axis,value,n,micro_f1
0,anchor_regime,strict,237,0.877285
1,anchor_regime,paraphrase,207,0.925150
2,anchor_regime,drift,225,0.919075
3,literal,0,327,0.883593
4,literal,1,342,0.929469
5,specificity,0,285,0.904348
6,specificity,1,384,0.907131
7,consistent,0,357,0.897436
8,consistent,1,312,0.916318
9,k_mods,1,312,0.931677


In [30]:
# interaction breakdown helper

def interaction_f1(name, mask):
    mask = np.array(mask)
    return {"group": name, "n": int(mask.sum()), "micro_f1": micro_f1_subset(mask)}

rows = []

# 2-way interactions
for r in ["strict","paraphrase","drift"]:
    for lit in [0,1]:
        rows.append(interaction_f1(f"{r} & literal={lit}", (test_df["anchor_regime"].eq(r) & test_df["literal"].eq(lit))))

for r in ["strict","paraphrase","drift"]:
    for k in [1,2,3]:
        rows.append(interaction_f1(f"{r} & k_mods={k}", (test_df["anchor_regime"].eq(r) & test_df["k_mods"].eq(k))))

for lit in [0,1]:
    for k in [1,2,3]:
        rows.append(interaction_f1(f"literal={lit} & k_mods={k}", (test_df["literal"].eq(lit) & test_df["k_mods"].eq(k))))

# 3-way "hard mode"
rows.append(interaction_f1("drift & metaphor & k_mods=3",
                           (test_df["anchor_regime"].eq("drift") & test_df["literal"].eq(0) & test_df["k_mods"].eq(3))))

rows.append(interaction_f1("paraphrase & metaphor & k_mods=3",
                           (test_df["anchor_regime"].eq("paraphrase") & test_df["literal"].eq(0) & test_df["k_mods"].eq(3))))

rows.append(interaction_f1("strict & inconsistent",
                           (test_df["anchor_regime"].eq("strict") & test_df["consistent"].eq(0))))

inter_table = pd.DataFrame(rows).sort_values(["micro_f1", "n"], ascending=[True, False])
inter_table.head(25)


,group,n,micro_f1
8,strict & k_mods=3,39,0.802030
20,literal=1 & k_mods=3,30,0.823529
21,drift & metaphor & k_mods=3,21,0.833333
17,literal=0 & k_mods=3,63,0.841463
14,drift & k_mods=3,33,0.842105
1,strict & literal=1,114,0.856322
2,paraphrase & literal=0,81,0.864686
23,strict & inconsistent,135,0.877506
11,paraphrase & k_mods=3,21,0.884956
4,drift & literal=0,123,0.886486


In [31]:
# multi-label confusion diagnostics

from sklearn.metrics import confusion_matrix

for i, lab in enumerate(mlb.classes_):
    cm = confusion_matrix(Y_test_true[:, i], Y_test_pred[:, i])
    tn, fp, fn, tp = cm.ravel()
    print(f"\n{lab}")
    print("TN FP\nFN TP")
    print(cm)
    print("FP rate:", fp / (fp + tn + 1e-9), "FN rate:", fn / (fn + tp + 1e-9))



nociception
TN FP
FN TP
[[412   8]
 [ 34 215]]
FP rate: 0.019047619047573697 FN rate: 0.13654618473840746

pressure
TN FP
FN TP
[[390   9]
 [ 39 231]]
FP rate: 0.022556390977387077 FN rate: 0.14444444444390947

temperature
TN FP
FN TP
[[403  20]
 [ 52 194]]
FP rate: 0.047281323876956785 FN rate: 0.21138211382027894

vibration
TN FP
FN TP
[[308   7]
 [ 31 323]]
FP rate: 0.022222222222151675 FN rate: 0.08757062146867918


In [32]:
# hard subset evaluation = excluding explicit tokens

no_explicit = ~test_df["text"].str.lower().str.contains(r"\bpressure\b|\bvibration(s)?\b|\btemperature\b")
print("Fraction of test with NO explicit tokens:", no_explicit.mean())
print("Micro F1 on no-explicit subset:", micro_f1_subset(no_explicit))


Fraction of test with NO explicit tokens: 0.7772795216741405
Micro F1 on no-explicit subset: 0.8992718446601942


/var/folders/9z/8mkwm57j4rq2mzhmvzjhn2n80000gn/T/ipykernel_13399/2040089123.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  no_explicit = ~test_df["text"].str.lower().str.contains(r"\bpressure\b|\bvibration(s)?\b|\btemperature\b")


In [33]:
# qualitative failure analysis - inspecting the hardest examples

errors = []
for i in range(len(test_df)):
    t = set(test_df.loc[i, "true_list"])
    p = set(test_df.loc[i, "pred_list"])
    if t != p:
        errors.append(i)

len(errors), len(test_df)


(174, 669)

In [34]:
# sampling 20 error cases with metadata

sample_idx = np.random.default_rng(42).choice(errors, size=min(20, len(errors)), replace=False)

cols = ["text","labels","anchor_regime","literal","specificity","consistent","k_mods","true_list","pred_list"]
test_df.loc[sample_idx, cols]


,text,labels,anchor_regime,literal,specificity,consistent,k_mods,true_list,pred_list
470,The bruise feels worse when pressure is applie...,nociception+pressure,strict,1,0,0,2,"(nociception, pressure)","(pressure,)"
476,A sharp vibration pulses through my fingertips...,vibration+pressure,drift,1,1,0,2,"(pressure, vibration)","(vibration,)"
111,The coldish area pulsates yet somehow the temp...,temperature+vibration,strict,1,0,0,2,"(temperature, vibration)","(temperature,)"
638,A tepid tightness spreads across my lower back...,temperature+vibration+pressure,strict,1,1,1,3,"(pressure, temperature, vibration)","(vibration,)"
559,It feels like the gentle touch of sunlight on ...,temperature,paraphrase,0,0,1,1,"(temperature,)",()
495,"It feels like a sharp ache in my lower back, a...",nociception+pressure,strict,0,1,0,2,"(nociception, pressure)","(nociception,)"
44,A scalding pain spreads across my forearm but ...,nociception,strict,1,1,0,1,"(nociception,)","(nociception, temperature)"
34,A scalding pain spreads across my cold forearm...,nociception,strict,1,1,0,1,"(nociception,)","(nociception, temperature)"
401,My back feels like it's being pressed by a hot...,nociception+temperature,paraphrase,0,1,1,2,"(nociception, temperature)","(pressure, temperature)"
363,"The tight feeling in my chest is sharp, like a...",pressure,strict,0,1,0,1,"(pressure,)","(nociception,)"


In [35]:
# example: temp missed 

temp_missed = []
temp_i = list(mlb.classes_).index("temperature")
for i in range(len(test_df)):
    if Y_test_true[i, temp_i] == 1 and Y_test_pred[i, temp_i] == 0:
        temp_missed.append(i)

test_df.loc[temp_missed[:20], cols]


,text,labels,anchor_regime,literal,specificity,consistent,k_mods,true_list,pred_list
21,My left thigh feels steady and still like ice ...,temperature+vibration,paraphrase,0,1,0,2,"(temperature, vibration)","(vibration,)"
23,My wrist feels warming yet buzzing like a phon...,temperature+vibration,paraphrase,0,1,0,2,"(temperature, vibration)","(vibration,)"
55,My wrist feels sore and vibrating but the heat...,nociception+temperature+vibration,strict,1,1,0,3,"(nociception, temperature, vibration)","(nociception, vibration)"
56,My elbow feels sore and vibrating but the heat...,nociception+temperature+vibration,strict,1,1,0,3,"(nociception, temperature, vibration)","(nociception,)"
109,The coldish sensation pulsates yet everything ...,temperature+vibration,strict,1,0,0,2,"(temperature, vibration)","(vibration,)"
110,The coldish feeling pulsates yet stays complet...,temperature+vibration,strict,1,0,0,2,"(temperature, vibration)","(vibration,)"
112,The coldish sensation pulsates yet remains com...,temperature+vibration,strict,1,0,0,2,"(temperature, vibration)","(vibration,)"
115,My left forearm had steady shaking like a moto...,temperature+vibration,paraphrase,0,1,0,2,"(temperature, vibration)","(vibration,)"
117,My forearm feels chilled yet buzzing steadily ...,temperature+vibration,paraphrase,0,1,0,2,"(temperature, vibration)","(vibration,)"
218,It feels like chilling pressure wrapping aroun...,temperature+pressure,strict,0,0,1,2,"(pressure, temperature)","(pressure,)"


## Second part of the evaluation: Cross-modal salience and its' influence on classification

In [36]:
# loading strict only test set and model correctness

import pandas as pd
import numpy as np

strict_test = test_df[test_df["anchor_regime"].eq("strict")].copy()

# exact-match correctness (all labels correct for that sample)
strict_test["exact_match"] = strict_test.apply(
    lambda r: set(r["true_list"]) == set(r["pred_list"]), axis=1
)

strict_test[["anchors_planned","labels","exact_match"]].head()


,anchors_planned,labels,exact_match
0,FROSTY; FORCE,temperature+pressure,True
1,FROSTY; FORCE,temperature+pressure,True
2,FROSTY; FORCE,temperature+pressure,True
3,FROSTY; FORCE,temperature+pressure,False
4,FROSTY; FORCE,temperature+pressure,True


In [41]:
# loading lancaster sensorimotor norms and prepping lookup

lex_path = PROJECT_ROOT / "data" / "raw" / "Sensorimotor_norms_24Jan2026.csv"
lex = pd.read_csv(lex_path)

# normalize Word field for matching
lex["Word_norm"] = lex["Word"].astype(str).str.strip().str.upper()

# columns we need
cols = [
    "Word_norm",
    "Haptic.mean", "Visual.mean", "Auditory.mean", "Olfactory.mean", "Gustatory.mean", "Interoceptive.mean"
]
lex_small = lex[cols].copy()

# make a lookup dict row-wise
lex_lookup = lex_small.set_index("Word_norm").to_dict(orient="index")




In [42]:
# parsing anchors_planned and matching to lexicon

def parse_anchors_planned(s):
    if pd.isna(s) or str(s).strip() == "":
        return []
    return [a.strip().upper() for a in str(s).split(";") if a.strip()]

other_modal_cols = ["Visual.mean","Auditory.mean","Olfactory.mean","Gustatory.mean","Interoceptive.mean"]

def anchor_to_scores(anchor):
    a = anchor.strip().upper()
    if a in lex_lookup:
        d = lex_lookup[a]
        h = d["Haptic.mean"]
        other_max = max(d[c] for c in other_modal_cols)
        return {"anchor": a, "found": True, "haptic": h, "other_max": other_max, "dominance": h - other_max}

    # fallback for multiword anchors: split and try parts
    parts = a.replace("-", " ").split()
    part_rows = [lex_lookup[p] for p in parts if p in lex_lookup]

    if len(part_rows) == 0:
        return {"anchor": a, "found": False, "haptic": np.nan, "other_max": np.nan, "dominance": np.nan}

    # pick the part with highest haptic mean (conservative)
    best = max(part_rows, key=lambda r: r["Haptic.mean"])
    h = best["Haptic.mean"]
    other_max = max(best[c] for c in other_modal_cols)
    return {"anchor": a, "found": True, "haptic": h, "other_max": other_max, "dominance": h - other_max}

# expand anchors per row
strict_test["anchors_list"] = strict_test["anchors_planned"].apply(parse_anchors_planned)


In [43]:
# computing dominance aggregates per sample

def row_dominance_stats(anchor_list):
    rows = [anchor_to_scores(a) for a in anchor_list]
    doms = [r["dominance"] for r in rows if np.isfinite(r["dominance"])]
    found = sum(1 for r in rows if r["found"])
    total = len(anchor_list)

    if len(doms) == 0:
        return pd.Series({
            "n_anchors": total,
            "n_found": found,
            "dominance_mean": np.nan,
            "dominance_min": np.nan
        })

    return pd.Series({
        "n_anchors": total,
        "n_found": found,
        "dominance_mean": float(np.mean(doms)),
        "dominance_min": float(np.min(doms))
    })

strict_test[["n_anchors","n_found","dominance_mean","dominance_min"]] = strict_test["anchors_list"].apply(row_dominance_stats)
strict_test[["n_anchors","n_found","dominance_mean","dominance_min","exact_match"]].head()


,n_anchors,n_found,dominance_mean,dominance_min,exact_match
0,2.0,2.0,-0.0605,-0.7,True
1,2.0,2.0,-0.0605,-0.7,True
2,2.0,2.0,-0.0605,-0.7,True
3,2.0,2.0,-0.0605,-0.7,False
4,2.0,2.0,-0.0605,-0.7,True


In [44]:
# sanity check

coverage = strict_test["n_found"].sum() / strict_test["n_anchors"].sum()
coverage


np.float64(1.0)

In [50]:
## Actual analysis

strict_ok = strict_test.loc[strict_test["exact_match"] == True, "dominance_mean"].dropna()
strict_bad = strict_test.loc[strict_test["exact_match"] == False, "dominance_mean"].dropna()

summary = pd.DataFrame({
    "Group": ["Correctly classified", "Incorrectly classified"],
    "Mean dominance": [strict_ok.mean(), strict_bad.mean()],
    "Median dominance": [strict_ok.median(), strict_bad.median()],
    "N": [len(strict_ok), len(strict_bad)]
})

summary


,Group,Mean dominance,Median dominance,N
0,Correctly classified,0.374353,0.4450,158
1,Incorrectly classified,0.360787,0.2475,79


In [51]:
# nonparam test

from scipy.stats import mannwhitneyu

u_stat, p_val = mannwhitneyu(
    strict_ok,
    strict_bad,
    alternative="greater"
)

print("Mann–Whitney U test (greater dominance → correct classification)")
print(f"U statistic: {u_stat:.1f}")
print(f"p-value:     {p_val:.4f}")
print(f"N correct:   {len(strict_ok)}")
print(f"N incorrect: {len(strict_bad)}")



Mann–Whitney U test (greater dominance → correct classification)
U statistic: 7177.0
p-value:     0.0298
N correct:   158
N incorrect: 79


In [52]:
# effect size

effect_size = u_stat / (len(strict_ok) * len(strict_bad))
print(f"Rank-biserial effect size (U / (n1·n2)): {effect_size:.3f}")


Rank-biserial effect size (U / (n1·n2)): 0.575


In [49]:
# logreg

import statsmodels.api as sm

tmp = strict_test.dropna(subset=["dominance_mean"]).copy()
X = sm.add_constant(tmp["dominance_mean"])
y = tmp["exact_match"].astype(int)

m = sm.Logit(y, X).fit(disp=False)
m.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:            exact_match   No. Observations:                  237
Model:                          Logit   Df Residuals:                      235
Method:                           MLE   Df Model:                            1
Date:                Sat, 31 Jan 2026   Pseudo R-squ.:               7.337e-05
Time:                        16:24:28   Log-Likelihood:                -150.84
converged:                       True   LL-Null:                       -150.85
Covariance Type:            nonrobust   LLR p-value:                    0.8817
==================================================================================
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.6818      0.157      4.330      0.000       0.373       0.990
dominance_mean     0.0310      0.208      0.149      0.882      -0.378       0.440
==================================================================================
"""